# Epochs Count

Counts valid epochs per participant, task, run, and event type, then aggregates by
condition for the Load and Flow papers. The minimum-trial threshold (`MIN_TRIALS`) applies to the
**Load** paper only; for Flow, the lowest counts are reported for information (no threshold).

**`TRIALS`** selects which stimulus-locked trials are counted, matching the evoked files used
in the analyses:
- `"correct"` (primary): targets = hits only; standards and novels = correct rejections only
- `"alltrials"` (sensitivity): every epoch kept after bad segments and after-response rejection

The step 8 stimulus-locked epochs files contain **all** trials (incorrect ones labelled in the
metadata), so the correct-only count is taken from `response_correct`.
Response-locked counts (hits only) are the same for both settings.
For the passive cohort, every trial is correct, so both settings give identical counts.

**Output CSVs on NAS** (`outputs/measures/behavior/`):
- `{date}_{paradigm}_{trials}_epochs_count_raw.csv` — per run, per event
- `{date}_{paradigm}_{trials}_epochs_count_load.csv` — aggregated by load condition
- `{date}_{paradigm}_{trials}_epochs_count_flow.csv` — aggregated by flow condition
- `{date}_{paradigm}_{trials}_epochs_below_threshold.csv` — Load conditions below `MIN_TRIALS`

Last edit: 31.10.2026 @anguyen

In [ ]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path
from sys import platform
import sys

import pandas as pd
from mne import read_epochs
from mne_lsl.utils.logs import warn
from tqdm import tqdm

def find_repo_root(marker="core") -> Path:
    """Walk up from the current directory until a folder containing
    `marker` (e.g. core/) is found -- avoids hardcoding folder depth,
    which breaks every time notebooks get moved."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from participants import get_participants

# ==============================================================================
# CONFIGURATION
# ==============================================================================

PARADIGM   = "active"    # "active" | "passive"
TRIALS     = "correct"   # "correct" (primary) | "alltrials" (sensitivity)
MIN_TRIALS = 30          # minimum trials per condition — Load paper only
DATE_STR = datetime.today().strftime("%Y%m%d")
STUDY    = "Active" if PARADIGM == "active" else "Passive"

if platform == "win32":
    root    = Path(f"L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2")
else:
    root    = Path(f"/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2")

out_path = root / "outputs" / "measures" / "behavior"
out_path.mkdir(parents=True, exist_ok=True)

# Max epochs per task/event (theoretical maximum)
MAX_EPOCHS = {
    "oddball": {"standard": 288, "target": 36, "novel": 36, "response": 36},
    "UT":      {"standard": 800, "target": 100, "novel": 100, "response": 100},
}

TASKS      = ["oddball", "UT"]
RUNS       = [1, 2]
STIM_TYPES = ["standard", "target", "novel"] if PARADIGM == "passive" else ["standard", "target", "novel", "response"]

assert TRIALS in ("correct", "alltrials"), f"Unknown TRIALS: {TRIALS}"
print(f"Paradigm : {PARADIGM.upper()}")
print(f"Trials   : {TRIALS}  (Load threshold: {MIN_TRIALS})")
print(f"root : {root}")
print(f"Output   : {out_path}")

KNOWN_MISSING = {             # runs with no step 8 data (see EXCLUSIONS.md)
    ("P03", "oddball", 2),    # gel bridging on all electrodes
    ("P16", "oddball", 2),    # too noisy
    ("P75", "oddball", 1),    # recording missing
    ("S08", "oddball", 1),    # recording missing
    ("S39", "oddball", 1),    # protocol deviation (wrong script, no counting task)
}


In [ ]:
# ==============================================================================
# Helper — count epochs from the step 8 .fif files (each file read once per run)
# ==============================================================================

def count_run(root, uid, gid, task, run):
    """Return {stim: count or None} for one run.

    Stimulus-locked: counted from the metadata of the step 8 epochs file.
    With TRIALS == "correct", only trials with response_correct == True count.
    Response-locked: number of epochs in the response-locked file (hits only).
    """
    out = {stim: None for stim in STIM_TYPES}
    if (uid, task, run) in KNOWN_MISSING:
        return out
    stem    = f"sub-{uid}-{gid}_task-{task}_run-{run}"
    sub_dir = root / f"sub-{uid}-{gid}" / stem

    # --- stimulus-locked ---
    path = sub_dir / f"{stem}_step8_stimlocked-epo.fif"
    if not path.exists():
        warn(f"File not found: {path.name}")
    else:
        try:
            md = read_epochs(path, preload=False, verbose=False).metadata
            keep = md["response_correct"].astype(bool) if TRIALS == "correct" else True
            for stim in ["standard", "target", "novel"]:
                out[stim] = int(((md["event_name"] == stim) & keep).sum())
        except Exception as e:
            warn(f"Error: {stem} stim-locked -> {e}")

    # --- response-locked (active only, hits only) ---
    if "response" in STIM_TYPES:
        path = sub_dir / f"{stem}_step8_responselocked_bc-200-100-epo.fif"
        if not path.exists():
            warn(f"File not found: {path.name}")
        else:
            try:
                out["response"] = len(read_epochs(path, preload=False, verbose=False))
            except Exception as e:
                warn(f"Error: {stem} response-locked -> {e}")
    return out

print("Helper ready.")


In [ ]:
# ==============================================================================
# Load all participants (both papers — filter later)
# ==============================================================================

# Use union of all included participants for this paradigm
included_flow = set(get_participants("flow", PARADIGM).keys())
included_load = set(get_participants("load", PARADIGM).keys())
all_included  = included_flow | included_load

# Get gid mapping — use load as source (has more participants)
uid_to_gid = {**get_participants("flow", PARADIGM), **get_participants("load", PARADIGM)}

print(f"Flow : {len(included_flow)} participants")
print(f"Load : {len(included_load)} participants")
print(f"Total unique: {len(all_included)} participants")


In [ ]:
# ==============================================================================
# Count epochs — raw (per participant, task, run, event)
# ==============================================================================

rows = []
for uid in tqdm(sorted(all_included), desc="Participants"):
    gid = uid_to_gid[uid]
    for task in TASKS:
        for run in RUNS:
            counts = count_run(root, uid, gid, task, run)
            for stim in STIM_TYPES:
                n_max   = MAX_EPOCHS[task].get(stim)
                n_count = counts[stim]
                pct     = round(n_count / n_max * 100, 2) if (n_count is not None and n_max) else None
                rows.append({
                    "participant": uid,
                    "task":        task,
                    "run":         run,
                    "stim":        stim,
                    "trials":      TRIALS,
                    "n_epochs":    n_count,
                    "n_max":       n_max,
                    "pct":         pct,
                    "in_flow":     uid in included_flow,
                    "in_load":     uid in included_load,
                })

df_raw = pd.DataFrame(rows)
print(f"Raw counts: {len(df_raw)} rows")
print(df_raw.head(8))


In [ ]:
# ==============================================================================
# Aggregate by load condition (simple = oddball runs 1+2, complex = UT runs 1+2)
# ==============================================================================

df_load = (
    df_raw[df_raw["in_load"] & df_raw["n_epochs"].notna()]   # missing runs don't add to n_max
    .groupby(["participant", "task", "stim"])
    .agg(
        n_epochs = ("n_epochs", "sum"),
        n_max    = ("n_max",    "sum"),
        n_runs   = ("run",      "nunique"),
    )
    .reset_index()
)
df_load["condition"] = df_load["task"].map({"oddball": "Simple", "UT": "Complex"})
df_load["pct"]       = (df_load["n_epochs"] / df_load["n_max"] * 100).round(2)

print("Load aggregated:")
print(df_load.head(6))


In [ ]:
# ==============================================================================
# Aggregate by flow condition (high/low flow — based on gid mapping)
# ==============================================================================

# Map participant to high/low flow run
flow_participants = get_participants("flow", PARADIGM)

def get_flow_condition(uid, gid, task, run):
    if task != "UT":
        return None
    high_run = 2 if gid in ["G1", "G3", "G5", "G7"] else 1
    return "High Flow" if run == high_run else "Low Flow"

df_raw["flow_condition"] = df_raw.apply(
    lambda r: get_flow_condition(r["participant"], uid_to_gid[r["participant"]], r["task"], r["run"]),
    axis=1
)

df_flow = (
    df_raw[df_raw["in_flow"] & df_raw["flow_condition"].notna()]
    .groupby(["participant", "flow_condition", "stim"])
    .agg(
        n_epochs = ("n_epochs", "sum"),
        n_max    = ("n_max",    "sum"),
    )
    .reset_index()
    .rename(columns={"flow_condition": "condition"})
)
df_flow["pct"] = (df_flow["n_epochs"] / df_flow["n_max"] * 100).round(2)


In [ ]:
# ------------------------------------------------------------------------------
# Two collapsed aggregations for flow, serving different reporting purposes:
#
# "Collapsed (load)": per-participant sum across both UT runs (High + Low Flow).
#   Each participant contributes one data point. Used in the load paper where
#   both UT runs together constitute the Complex condition. This is the
#   statistically correct unit of analysis when the participant is the unit.
#
# "Collapsed (flow)": all runs pooled as independent observations (N=72 rows).
#   Each run contributes one data point. Used in the flow paper where High and
#   Low Flow are separate conditions — per-run percentages reflect the actual
#   data available per condition. Matches the values submitted in the flow paper.
# ------------------------------------------------------------------------------

# Collapsed (load) — per-participant, both runs summed
df_flow_collapsed_load = (
    df_flow[df_flow["condition"].isin(["High Flow", "Low Flow"])]
    .groupby(["participant", "stim"])
    .agg(n_epochs=("n_epochs", "sum"), n_max=("n_max", "sum"))
    .reset_index()
)
df_flow_collapsed_load["condition"] = "Collapsed (load)"
df_flow_collapsed_load["pct"] = (
    df_flow_collapsed_load["n_epochs"] / df_flow_collapsed_load["n_max"] * 100
).round(2)

# Collapsed (flow) — per-run pooled, each run as independent observation
# N = 72 (36 participants x 2 runs). Matches submitted flow paper values.
df_flow_collapsed_flow = df_flow[df_flow["condition"].isin(["High Flow", "Low Flow"])].copy()
df_flow_collapsed_flow["condition"] = "Collapsed (flow)"

df_flow = pd.concat(
    [df_flow, df_flow_collapsed_load, df_flow_collapsed_flow],
    ignore_index=True
)

In [ ]:
# ==============================================================================
# Save CSVs on NAS
# ==============================================================================

fname_raw  = out_path / f"{DATE_STR}_{PARADIGM}_{TRIALS}_epochs_count_raw.csv"
fname_load = out_path / f"{DATE_STR}_{PARADIGM}_{TRIALS}_epochs_count_load.csv"
fname_flow = out_path / f"{DATE_STR}_{PARADIGM}_{TRIALS}_epochs_count_flow.csv"

df_raw.to_csv(fname_raw,   index=False)
df_load.to_csv(fname_load, index=False)
df_flow.to_csv(fname_flow, index=False)

print(f"Saved -> {fname_raw.name}")
print(f"Saved -> {fname_load.name}")
print(f"Saved -> {fname_flow.name}")


In [ ]:
# ==============================================================================
# Report — APA sentences (percentage + raw count together)
# ==============================================================================
# Raw counts alone aren't directly comparable across stimulus types -- standard
# has a much larger denominator (288/800) than target/novel/response (36/100),
# so a bare count like "251 standard" sitting next to "35 target" looks like a
# large, uninterpretable gap. Reporting both, with the max stated once per
# stimulus type, keeps the count interpretable without needing a separate table.

def epochs_report(df, paper, label):
    print(f"=== {paper.upper()} - {label.upper()} - {TRIALS} trials ===")
    conditions = (
        ["Simple", "Complex"]
        if paper == "load"
        else ["High Flow", "Low Flow", "Collapsed (load)", "Collapsed (flow)"]
    )
    for condition in conditions:
        print(f"  -- {condition} --")
        # Note: Collapsed (load) = per-participant sum across both UT runs (N=participants)
        # Note: Collapsed (flow) = per-run pool across both conditions (N=participants x 2)
        df_cond = df[df["condition"] == condition]
        for stim in sorted(df_cond["stim"].unique()):
            df_s = df_cond[df_cond["stim"] == stim].dropna(subset=["pct"])
            if len(df_s) == 0:
                continue
            stim_label = "deviant" if stim == "target" and PARADIGM == "passive" else stim
            n_max = int(df_s["n_max"].iloc[0])  # constant within a stim/condition group
            print(
                f"    {stim_label} (max {n_max}): "
                f"count range {int(df_s['n_epochs'].min())}-{int(df_s['n_epochs'].max())} "
                f"(M = {df_s['n_epochs'].mean():.1f}, SD = {df_s['n_epochs'].std():.1f}); "
                f"{df_s['pct'].min():.1f}%-{df_s['pct'].max():.1f}% "
                f"(M = {df_s['pct'].mean():.2f}%, SD = {df_s['pct'].std():.2f}%)"
                f" [N={len(df_s)}]"
            )
    print()

epochs_report(df_load, "load", PARADIGM)
epochs_report(df_flow, "flow", PARADIGM)


In [ ]:
# ==============================================================================
# Save APA sentences to text file (percentage + raw count together)
# ==============================================================================

STIM_ORDER = ["target", "standard", "novel", "response"]

out_txt = out_path / f"{DATE_STR}_{PARADIGM}_{TRIALS}_epochs_count_report.txt"

with open(out_txt, "w") as f:
    for df, paper, label in [(df_load, "load", PARADIGM), (df_flow, "flow", PARADIGM)]:
        f.write(f"=== {paper.upper()} - {label.upper()} - {TRIALS} trials ===\n\n")

        if paper == "load":
            f.write(
                "Note: epochs aggregated across 2 runs per condition "
                "(Simple = oddball run1+run2, Complex = UT run1+run2).\n"
                "Unit of analysis: participant (N=participants).\n"
                "Percentages and counts are comparable within each condition "
                "(fixed denominator per condition: "
                "Simple: 576 standard, 72 target, 72 novel; "
                "Complex: 1600 standard, 200 target, 200 novel).\n\n"
            )
            conditions = ["Simple", "Complex"]
        else:
            f.write(
                "Note: High Flow and Low Flow each correspond to a single UT run "
                "(100 target, 100 novel, 800 standard).\n"
                "Collapsed (load): per-participant sum across both runs "
                "(N=participants). Use for load paper Complex condition reporting.\n"
                "Collapsed (flow): per-run pool across both conditions "
                "(N=participants x 2). Matches submitted flow paper values, "
                "where each run is an independent observation reflecting "
                "data available per condition.\n\n"
            )
            conditions = ["High Flow", "Low Flow", "Collapsed (load)", "Collapsed (flow)"]

        stim_available = df["stim"].unique()
        ordered_stims  = [s for s in STIM_ORDER if s in stim_available]

        for condition in conditions:
            f.write(f"-- {condition} --\n")
            # Collapsed (load): N=participants (per-participant aggregation)
            # Collapsed (flow): N=participants x 2 (per-run pooling, matches flow paper)
            df_cond = df[df["condition"] == condition]
            for stim in ordered_stims:
                df_s       = df_cond[df_cond["stim"] == stim].dropna(subset=["pct"])
                stim_label = "deviant" if stim == "target" and PARADIGM == "passive" else stim
                if len(df_s) == 0:
                    continue
                n_max = int(df_s["n_max"].iloc[0])
                f.write(
                    f"[N={len(df_s)}] "
                    f"For the {stim_label} stimuli (maximum {n_max} epochs), the number of "
                    f"epochs retained ranged from {int(df_s['n_epochs'].min())} to "
                    f"{int(df_s['n_epochs'].max())} (M = {df_s['n_epochs'].mean():.1f}, "
                    f"SD = {df_s['n_epochs'].std():.1f}), corresponding to "
                    f"{df_s['pct'].min():.1f}% to {df_s['pct'].max():.1f}% of epochs retained "
                    f"(M = {df_s['pct'].mean():.2f}%, SD = {df_s['pct'].std():.2f}%).\n"
                )
            f.write("\n")

print(f"Saved -> {out_txt.name}")


In [ ]:
# ==============================================================================
# Summary table for paper
# ==============================================================================
out_csv    = out_path / f"{DATE_STR}_{PARADIGM}_{TRIALS}_epochs_summary_table.csv"
STIM_ORDER = ["target", "standard", "novel", "response"]

summary = (
    df_load[df_load["stim"].isin(STIM_ORDER)]
    .groupby(["condition", "stim"])["pct"]
    .agg(M="mean", SD="std", Min="min", Max="max")
    .round(2)
    .reset_index()
)

summary_wide = summary.pivot(index="stim", columns="condition", values=["M", "SD", "Min", "Max"])
summary_wide = summary_wide.reindex(STIM_ORDER)

df_table = pd.DataFrame({
    "Stimulus":        [s.capitalize() for s in summary_wide.index],
    "Simple M (SD)":   [f"{summary_wide['M']['Simple'][s]:.2f}% ({summary_wide['SD']['Simple'][s]:.2f}%)" for s in summary_wide.index],
    "Simple Range":    [f"{summary_wide['Min']['Simple'][s]:.1f}-{summary_wide['Max']['Simple'][s]:.1f}%" for s in summary_wide.index],
    "Complex M (SD)":  [f"{summary_wide['M']['Complex'][s]:.2f}% ({summary_wide['SD']['Complex'][s]:.2f}%)" for s in summary_wide.index],
    "Complex Range":   [f"{summary_wide['Min']['Complex'][s]:.1f}-{summary_wide['Max']['Complex'][s]:.1f}%" for s in summary_wide.index],
})

df_table.to_csv(out_csv, index=False)
print(df_table.to_string(index=False))
print(f"\nSaved -> {out_csv.name}")

In [ ]:
print("\nNote for table caption:")
print("Simple: 2×oddball runs (max 576 standard, 72 deviant, 72 novel epochs [+ 72 response in active]).")
print("Complex: 2×UT runs (max 1600 standard, 200 deviant, 200 novel epochs [+ 200 response in active]).")
print("Deviant stimuli are labeled 'target' in the active paradigm and 'deviant' in the passive paradigm.")
print("Response-locked epochs exist for the active paradigm only.")
if TRIALS == "correct":
    print("Stimulus-locked counts include correct trials only (targets: hits; standards/novels: correct rejections).")
else:
    print("Stimulus-locked counts include all trials (correct and incorrect).")
print("Runs missing or excluded (see EXCLUSIONS.md) do not count toward the maximum.")
print(f"N = {df_load['participant'].nunique()} participants (Load), "
      f"{df_flow['participant'].nunique()} participants (Flow) — {PARADIGM}.")


In [ ]:
# ==============================================================================
# Minimum-trial check — Load only (threshold MIN_TRIALS, runs summed: Simple / Complex)
# Flow has no threshold: its lowest counts are listed for reporting only.
# ==============================================================================

below_load = df_load[df_load["n_epochs"] < MIN_TRIALS].assign(paper="load")

print(f"=== LOAD: conditions below {MIN_TRIALS} trials — {PARADIGM}, {TRIALS} ===")
print(f"{len(below_load)} condition(s), {below_load['participant'].nunique()} participant(s)")
if len(below_load):
    print(below_load[["participant", "condition", "stim", "n_epochs", "n_max"]].to_string(index=False))

fname_below = out_path / f"{DATE_STR}_{PARADIGM}_{TRIALS}_epochs_below_threshold.csv"
below_load.to_csv(fname_below, index=False)
print(f"Saved -> {fname_below.name}")

# Flow — information only (no threshold): lowest count per stimulus type, per flow condition
flow_runs = df_flow[df_flow["condition"].isin(["High Flow", "Low Flow"])].dropna(subset=["n_epochs"])
lowest = flow_runs.loc[flow_runs.groupby(["condition", "stim"])["n_epochs"].idxmin()]
print(f"\n=== FLOW (no threshold): lowest count per condition and stimulus — {PARADIGM}, {TRIALS} ===")
print(lowest[["condition", "stim", "participant", "n_epochs", "n_max"]].to_string(index=False))